# Clase 15 — Práctica: Proyecto integrador (ETL de cierre)

Esta es la práctica de cierre del curso. En vez de ejercicios sueltos, vas a resolver **dos mini-proyectos** que replican el mismo flujo **Extract → Transform → Load** que viste en la clase teórica, pero sobre datasets distintos al del PIB (ventas y población), definidos directamente en el notebook para que no dependas de ningún archivo externo ni de internet.

Para cada ejercicio:
1. Lee el enunciado y las pistas.
2. Completa el código donde dice `# TODO`.
3. Ejecuta la celda y compara el resultado con lo esperado en el enunciado.
4. Si te trabas, despliega la solución comentada debajo — pero intenta primero sin mirarla.

In [ ]:
import numpy as np
import pandas as pd

## Ejercicio 1 — ETL de ventas de una tienda online

Trabajas como analista de datos junior para una tienda online. Te dan un registro de ventas de la última semana con: producto, unidades vendidas y precio unitario en dólares. Te piden un reporte limpio con el **ingreso total por producto en miles de dólares** (redondeado a 2 decimales), guardado en un CSV.

Los datos de partida (ya cargados como diccionario, simulando lo que sería un `pd.read_csv()` o un `pd.read_html()`):

```python
datos_ventas = {
    'Producto': ['Auriculares', 'Teclado', 'Mouse', 'Monitor', 'Webcam', 'Micrófono'],
    'Unidades': ['120', '85', '200', '40', '95', '60'],   # llegan como texto, típico de un scraping
    'PrecioUnitario': [25.5, 45.0, 12.0, 180.0, 30.0, 55.0],
}
```

Completa el flujo ETL:

1. **Extract**: convierte el diccionario en un DataFrame (`pd.DataFrame(...)`).
2. **Transform**:
   - Selecciona con `.loc` solo las filas de los primeros 4 productos (pista: `.loc[0:3]` incluye el límite superior).
   - Convierte la columna `Unidades` a entero con `.astype(int)` (viene como texto).
   - Crea una nueva columna `IngresoTotal` = `Unidades * PrecioUnitario` (operación vectorizada, sin bucle).
   - Convierte `IngresoTotal` a miles de dólares (divide entre 1000) y redondea a 2 decimales con `np.round()`.
3. **Load**: exporta el resultado a `reporte_ventas.csv` con `.to_csv()`.

**Pistas:**
- `.loc[a:b]` selecciona por **etiqueta** e incluye ambos extremos; `.iloc[a:b]` selecciona por **posición** y excluye el final — no son intercambiables.
- Una columna nueva se crea asignando directamente: `df['NuevaColumna'] = ...`.
- `np.round(valor, 2)` funciona igual sobre un array, una Serie o una columna de DataFrame.

In [ ]:
datos_ventas = {
    'Producto': ['Auriculares', 'Teclado', 'Mouse', 'Monitor', 'Webcam', 'Micrófono'],
    'Unidades': ['120', '85', '200', '40', '95', '60'],   # llegan como texto, típico de un scraping
    'PrecioUnitario': [25.5, 45.0, 12.0, 180.0, 30.0, 55.0],
}

# TODO: (Extract) convierte el diccionario en un DataFrame
df_ventas = None

# TODO: (Transform 1) selecciona con .loc solo las filas de los primeros 4 productos
df_ventas = None

# TODO: (Transform 2) convierte la columna 'Unidades' a entero con .astype(int)


# TODO: (Transform 3) crea la columna 'IngresoTotal' = Unidades * PrecioUnitario


# TODO: (Transform 4) convierte 'IngresoTotal' a miles de dólares (/1000) y redondea a 2
#       decimales con np.round(). Puedes hacerlo en uno o dos pasos.


# TODO: (Load) exporta df_ventas a 'reporte_ventas.csv' con .to_csv()


print(df_ventas)

<details>
<summary>💡 Ver solución</summary>

```python
datos_ventas = {
    'Producto': ['Auriculares', 'Teclado', 'Mouse', 'Monitor', 'Webcam', 'Micrófono'],
    'Unidades': ['120', '85', '200', '40', '95', '60'],
    'PrecioUnitario': [25.5, 45.0, 12.0, 180.0, 30.0, 55.0],
}

# Extract: el diccionario se convierte en DataFrame, tal como haría pd.read_html() con una tabla real
df_ventas = pd.DataFrame(datos_ventas)

# Transform 1: .loc[0:3] selecciona por ETIQUETA e incluye el límite superior -> 4 filas (0,1,2,3)
df_ventas = df_ventas.loc[0:3]

# Transform 2: la columna llega como texto ('120'), la convertimos a entero para poder operar
df_ventas['Unidades'] = df_ventas['Unidades'].astype(int)

# Transform 3: operación aritmética vectorizada entre dos columnas, sin ningún bucle for
df_ventas['IngresoTotal'] = df_ventas['Unidades'] * df_ventas['PrecioUnitario']

# Transform 4: pasamos a miles de dólares y redondeamos con NumPy, como en el proyecto del PIB
df_ventas['IngresoTotal'] = np.round(df_ventas['IngresoTotal'] / 1000, 2)

# Load: dejamos el resultado listo para que otra persona (o tú mismo) lo reutilice
df_ventas.to_csv('reporte_ventas.csv')

print(df_ventas)
```
</details>

## Ejercicio 2 — ETL de población mundial por continente

Ahora vas a repetir el mismo flujo con otro dataset: población estimada (en millones de habitantes) de varios países, junto con su continente. El encargo: quedarte solo con los países de **Asia** y **América**, calcular qué **porcentaje representa cada país sobre el total combinado de esos dos continentes**, y guardar el resultado ordenado de mayor a menor población.

Los datos de partida:

```python
datos_poblacion = {
    'Pais': ['China', 'India', 'Brasil', 'Estados Unidos', 'Nigeria', 'Indonesia', 'México'],
    'Continente': ['Asia', 'Asia', 'América', 'América', 'África', 'Asia', 'América'],
    'PoblacionMillones': ['1412', '1408', '216', '335', '223', '277', '128'],  # texto, típico de scraping
}
```

Completa el flujo ETL:

1. **Extract**: convierte el diccionario en un DataFrame.
2. **Transform**:
   - Convierte `PoblacionMillones` a entero con `.astype(int)`.
   - Filtra con **selección booleana** (no `.iloc` ni `.loc` con números) las filas donde `Continente` sea `'Asia'` **o** `'América'`. Pista: usa `.isin(['Asia', 'América'])`.
   - Ordena el resultado de mayor a menor población con `.sort_values(..., ascending=False)`.
   - Calcula una columna `PorcentajeDelTotal` = `PoblacionMillones / suma_total * 100`, redondeada a 2 decimales con `np.round()`. La suma total la calculas con `.sum()` **antes** de crear la columna.
3. **Load**: exporta el resultado a `reporte_poblacion.csv`.

**Pistas:**
- `df[condición]` con una condición booleana es la forma más común de filtrar filas en Pandas — es el mismo mecanismo que ya usaste en clases anteriores con `df[df['columna'] > valor]`.
- `.isin([...])` te evita encadenar varios `|` (OR) cuando quieres comparar contra una lista de valores.
- Guarda la suma total en una variable **antes** de filtrar más la tabla, para no calcularla sobre datos ya recortados.

In [ ]:
datos_poblacion = {
    'Pais': ['China', 'India', 'Brasil', 'Estados Unidos', 'Nigeria', 'Indonesia', 'México'],
    'Continente': ['Asia', 'Asia', 'América', 'América', 'África', 'Asia', 'América'],
    'PoblacionMillones': ['1412', '1408', '216', '335', '223', '277', '128'],  # texto, típico de scraping
}

# TODO: (Extract) convierte el diccionario en un DataFrame
df_poblacion = None

# TODO: (Transform 1) convierte 'PoblacionMillones' a entero con .astype(int)


# TODO: (Transform 2) filtra con selección booleana (.isin) las filas de Asia o América
df_poblacion = None

# TODO: (Transform 3) ordena de mayor a menor población con .sort_values()
df_poblacion = None

# TODO: (Transform 4) calcula el porcentaje de cada país sobre el total combinado
#       (suma total = df_poblacion['PoblacionMillones'].sum(), calculada sobre la tabla YA filtrada)
total_asia_america = None
df_poblacion['PorcentajeDelTotal'] = None

# TODO: (Load) exporta df_poblacion a 'reporte_poblacion.csv'


print(df_poblacion)

<details>
<summary>💡 Ver solución</summary>

```python
datos_poblacion = {
    'Pais': ['China', 'India', 'Brasil', 'Estados Unidos', 'Nigeria', 'Indonesia', 'México'],
    'Continente': ['Asia', 'Asia', 'América', 'América', 'África', 'Asia', 'América'],
    'PoblacionMillones': ['1412', '1408', '216', '335', '223', '277', '128'],
}

# Extract: diccionario -> DataFrame
df_poblacion = pd.DataFrame(datos_poblacion)

# Transform 1: la población llega como texto, la convertimos a entero para poder operar y ordenar
df_poblacion['PoblacionMillones'] = df_poblacion['PoblacionMillones'].astype(int)

# Transform 2: selección booleana con .isin() -> nos quedamos solo con Asia y América
#             (equivalente a: (df_poblacion['Continente'] == 'Asia') | (df_poblacion['Continente'] == 'América'))
df_poblacion = df_poblacion[df_poblacion['Continente'].isin(['Asia', 'América'])]

# Transform 3: ordenamos de mayor a menor población
df_poblacion = df_poblacion.sort_values('PoblacionMillones', ascending=False)

# Transform 4: la suma total se calcula sobre la tabla YA filtrada (solo Asia + América),
# igual que en el proyecto del PIB, donde primero recortamos y después transformamos.
total_asia_america = df_poblacion['PoblacionMillones'].sum()
df_poblacion['PorcentajeDelTotal'] = np.round(
    df_poblacion['PoblacionMillones'] / total_asia_america * 100, 2
)

# Load: guardamos el resultado final
df_poblacion.to_csv('reporte_poblacion.csv')

print(df_poblacion)
```
</details>

## Cierre

Si completaste los dos ejercicios, acabas de resolver dos veces el mismo ciclo **Extract → Transform → Load** que viste en la clase teórica con el proyecto del PIB, pero con datasets y reglas de negocio distintas: ventas de una tienda y población por continente. Ese es exactamente el punto de esta última clase — el flujo de trabajo (extraer, dar forma con `.iloc`/`.loc` o selección booleana, corregir tipos con `.astype()`, operar con NumPy, y guardar con `.to_csv()`) se repite una y otra vez, sin importar el dominio del problema.

¡Felicitaciones por llegar al final del curso!